In [1]:
import pyspark
from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pandas as pd

# This CATALOG_URL works for the "docker compose" testing and development environment
# Change 'lakekeeper' if you are not running on "docker compose" (f. ex. 'localhost' if Lakekeeper is running locally).
CATALOG_URL = "http://lakekeeper:8181/catalog"
WAREHOUSE = "demo"

SPARK_VERSION = pyspark.__version__
SPARK_MINOR_VERSION = '.'.join(SPARK_VERSION.split('.')[:2])
ICEBERG_VERSION = "1.6.1"

# Connect with Spark

In [2]:
config = {
    f"spark.sql.catalog.lakekeeper": "org.apache.iceberg.spark.SparkCatalog",
    f"spark.sql.catalog.lakekeeper.type": "rest",
    f"spark.sql.catalog.lakekeeper.uri": CATALOG_URL,
    f"spark.sql.catalog.lakekeeper.warehouse": WAREHOUSE,
    f"spark.sql.catalog.lakekeeper.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.defaultCatalog": "lakekeeper",
    "spark.jars.packages": f"org.apache.iceberg:iceberg-spark-runtime-{SPARK_MINOR_VERSION}_2.12:{ICEBERG_VERSION},org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}",
}


In [3]:
spark_config = SparkConf().setMaster('local').setAppName("Iceberg-REST")
for k, v in config.items():
    spark_config = spark_config.set(k, v)

spark = SparkSession.builder.config(conf=spark_config).getOrCreate()

spark.sql("USE lakekeeper")

DataFrame[]

## Read and Write Tables

In [4]:
spark.sql("SHOW NAMESPACES IN lakekeeper").show(truncate=False)

+---------+
|namespace|
+---------+
|bronze   |
|silver   |
|gold     |
+---------+



In [5]:
spark.sql("SHOW NAMESPACES IN lakekeeper").show(truncate=False)

+---------+
|namespace|
+---------+
|bronze   |
|silver   |
|gold     |
+---------+



In [6]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM lakekeeper.bronze.part
""").show()

+---------+
|row_count|
+---------+
|   200000|
+---------+



In [7]:
spark.sql("""
    SELECT *
    FROM lakekeeper.gold.q16_result
    ORDER BY supplier_cnt DESC, brand, type, size
    LIMIT 10
""").show(truncate=False)

+--------+------------------------+----+------------+
|brand   |type                    |size|supplier_cnt|
+--------+------------------------+----+------------+
|Brand#41|MEDIUM BRUSHED TIN      |3   |28          |
|Brand#54|STANDARD BRUSHED COPPER |14  |27          |
|Brand#11|STANDARD BRUSHED TIN    |23  |24          |
|Brand#11|STANDARD BURNISHED BRASS|36  |24          |
|Brand#15|MEDIUM ANODIZED NICKEL  |3   |24          |
|Brand#15|SMALL ANODIZED BRASS    |45  |24          |
|Brand#15|SMALL BURNISHED NICKEL  |19  |24          |
|Brand#21|MEDIUM ANODIZED COPPER  |3   |24          |
|Brand#22|SMALL BRUSHED NICKEL    |3   |24          |
|Brand#22|SMALL BURNISHED BRASS   |19  |24          |
+--------+------------------------+----+------------+



In [ ]:
spark.sql(f"SELECT * FROM my_namespace.my_table").toPandas()

In [8]:
spark.sql("""
    CREATE TABLE lakekeeper.gold.iceberg_features_demo
    USING iceberg
    AS SELECT brand, type, size
    FROM lakekeeper.bronze.part
    LIMIT 100
""")

DataFrame[]

In [9]:
spark.sql("""
    ALTER TABLE lakekeeper.gold.iceberg_features_demo
    ADD COLUMN schema_demo STRING
""")

spark.sql("""
    DESCRIBE lakekeeper.gold.iceberg_features_demo
""").show(truncate=False)

+-----------+---------+-------+
|col_name   |data_type|comment|
+-----------+---------+-------+
|brand      |string   |NULL   |
|type       |string   |NULL   |
|size       |int      |NULL   |
|schema_demo|string   |NULL   |
+-----------+---------+-------+



In [10]:
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakekeeper.gold.iceberg_features_demo.snapshots
""").show(truncate=False)

+------------------+-----------------------+---------+
|snapshot_id       |committed_at           |operation|
+------------------+-----------------------+---------+
|857657110169374553|2026-10-17 12:58:51.843|append   |
+------------------+-----------------------+---------+



In [11]:
spark.sql("""
    INSERT INTO lakekeeper.gold.iceberg_features_demo
    VALUES ('TEST_BRAND', 'TEST_TYPE', 999, 'time-travel-test')
""")

spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM lakekeeper.gold.iceberg_features_demo.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1822114596689348290|2026-10-17 13:00:32.448|append   |
|857657110169374553 |2026-10-17 12:58:51.843|append   |
+-------------------+-----------------------+---------+



In [12]:
old_snapshot = 857657110169374553

spark.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM lakekeeper.gold.iceberg_features_demo
    VERSION AS OF {old_snapshot}
""").show()

+---------+
|row_count|
+---------+
|      100|
+---------+



In [13]:
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM lakekeeper.gold.iceberg_features_demo
""").show()

+---------+
|row_count|
+---------+
|      101|
+---------+



In [ ]:
33